# Ingestion Task — Download External Sources to UC Volume

This notebook runs as **Task 1** of the `tel_aviv_compensation_job` before the pipeline refresh.
It downloads street closures (CSV) and businesses (ArcGIS API) into a UC Volume,
where Auto Loader picks them up incrementally.

**Parameters** (passed from the job or overridden via widgets):
- `volume_path` — UC Volume base path
- `csv_url` — Street closures CSV download URL
- `api_base_url` — ArcGIS businesses API endpoint
- `batch_size` — API pagination batch size

In [0]:
import requests
import json
import os
import time
import logging
from datetime import datetime
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ---------------------------------------------------------------------------
# Widget parameters — overridable from the job task or interactive runs
# ---------------------------------------------------------------------------
dbutils.widgets.text("volume_path", "/Volumes/main/tel_aviv_dev/raw_data", "UC Volume base path")
dbutils.widgets.text("csv_url", "https://storage.googleapis.com/test_onedatai/rechov_sagur.csv", "Closures CSV URL")
dbutils.widgets.text("api_base_url", "https://gisn.tel-aviv.gov.il/arcgis/rest/services/IView2/MapServer/925/query", "Businesses API URL")
dbutils.widgets.text("batch_size", "2000", "API pagination batch size")

VOLUME_BASE = dbutils.widgets.get("volume_path")
CSV_URL = dbutils.widgets.get("csv_url")
API_BASE_URL = dbutils.widgets.get("api_base_url")
BATCH_SIZE = int(dbutils.widgets.get("batch_size"))

CLOSURES_DIR = f"{VOLUME_BASE}/street_closures"
BUSINESSES_DIR = f"{VOLUME_BASE}/businesses"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# ---------------------------------------------------------------------------
# Logging — structured output for job run diagnostics
# ---------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("ingest")

# ---------------------------------------------------------------------------
# HTTP session with automatic retries (3 retries, backoff on 429/500/502/503)
# ---------------------------------------------------------------------------
session = requests.Session()
retry_strategy = Retry(
    total=3,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
)
session.mount("https://", HTTPAdapter(max_retries=retry_strategy))
session.mount("http://", HTTPAdapter(max_retries=retry_strategy))

# Ensure directories exist
os.makedirs(CLOSURES_DIR, exist_ok=True)
os.makedirs(BUSINESSES_DIR, exist_ok=True)

logger.info(f"Volume base: {VOLUME_BASE}")
logger.info(f"Timestamp: {timestamp}")
logger.info(f"Batch size: {BATCH_SIZE}")

2026-10-16 13:47:56,927 [INFO] Volume base: /Volumes/main/tel_aviv_dev/raw_data
2026-10-16 13:47:56,928 [INFO] Timestamp: 20261016_134752
2026-10-16 13:47:56,930 [INFO] Batch size: 2000


In [0]:
logger.info(f"Downloading street closures CSV from {CSV_URL}...")
try:
    resp = session.get(CSV_URL, timeout=120)
    resp.raise_for_status()
    resp.encoding = "utf-8"

    closures_path = f"{CLOSURES_DIR}/closures_{timestamp}.csv"
    with open(closures_path, "w", encoding="utf-8") as f:
        f.write(resp.text)

    closures_bytes = len(resp.text)
    logger.info(f"Saved {closures_bytes:,} bytes to {closures_path}")
except Exception as e:
    logger.error(f"Failed to download closures CSV: {e}")
    dbutils.notebook.exit(json.dumps({"status": "FAILED", "error": f"Closures download failed: {e}"}))

2026-10-16 13:48:25,492 [INFO] Downloading street closures CSV from https://storage.googleapis.com/test_onedatai/rechov_sagur.csv...
2026-10-16 13:48:26,631 [INFO] Saved 249,538 bytes to /Volumes/main/tel_aviv_dev/raw_data/street_closures/closures_20261016_134752.csv


In [0]:
logger.info(f"Downloading businesses from ArcGIS API...")
offset = 0
total_features = 0
business_paths = []

try:
    while True:
        params = {
            "where": "1=1",
            "outFields": "*",
            "f": "json",
            "resultOffset": offset,
            "resultRecordCount": BATCH_SIZE,
        }
        resp = session.get(API_BASE_URL, params=params, timeout=120)
        resp.raise_for_status()
        data = resp.json()
        features = data.get("features", [])
        if not features:
            break

        out_path = f"{BUSINESSES_DIR}/businesses_{timestamp}_{offset}.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(features, f, ensure_ascii=False)

        total_features += len(features)
        business_paths.append(out_path)
        logger.info(f"Batch offset={offset}: {len(features)} features")

        if not data.get("exceededTransferLimit", False):
            break
        offset += BATCH_SIZE

    logger.info(f"Total features downloaded: {total_features:,}")
    logger.info(f"Files written: {len(business_paths)}")
except Exception as e:
    logger.error(f"Failed to download businesses at offset {offset}: {e}")
    dbutils.notebook.exit(json.dumps({"status": "FAILED", "error": f"Businesses download failed at offset {offset}: {e}"}))

2026-10-16 13:48:53,272 [INFO] Downloading businesses from ArcGIS API...
2026-10-16 13:48:55,788 [INFO] Batch offset=0: 2000 features
2026-10-16 13:48:58,707 [INFO] Batch offset=2000: 2000 features
2026-10-16 13:49:00,108 [INFO] Batch offset=4000: 2000 features
2026-10-16 13:49:01,600 [INFO] Batch offset=6000: 2000 features
2026-10-16 13:49:02,798 [INFO] Batch offset=8000: 2000 features
2026-10-16 13:49:04,442 [INFO] Batch offset=10000: 2000 features
2026-10-16 13:49:05,949 [INFO] Batch offset=12000: 2000 features
2026-10-16 13:49:07,092 [INFO] Batch offset=14000: 2000 features
2026-10-16 13:49:08,517 [INFO] Batch offset=16000: 2000 features
2026-10-16 13:49:11,324 [INFO] Batch offset=18000: 2000 features
2026-10-16 13:49:12,643 [INFO] Batch offset=20000: 2000 features
2026-10-16 13:49:13,461 [INFO] Batch offset=22000: 2000 features
2026-10-16 13:49:14,280 [INFO] Batch offset=24000: 2000 features
2026-10-16 13:49:15,573 [INFO] Batch offset=26000: 2000 features
2026-10-16 13:49:16,621 [

In [0]:
# ---------------------------------------------------------------------------
# Verify volume contents and exit with structured result
# ---------------------------------------------------------------------------
result = {"status": "SUCCESS", "timestamp": timestamp, "sources": {}}

for subdir in ["street_closures", "businesses"]:
    path = f"{VOLUME_BASE}/{subdir}"
    files = os.listdir(path)
    total_size = sum(os.path.getsize(f"{path}/{fn}") for fn in files)
    result["sources"][subdir] = {"files": len(files), "total_bytes": total_size}
    logger.info(f"{subdir}/: {len(files)} files ({total_size:,} bytes)")

result["closures_bytes"] = closures_bytes
result["businesses_features"] = total_features
result["businesses_batches"] = len(business_paths)

logger.info("Ingestion complete.")
logger.info(f"Result: {json.dumps(result, indent=2)}")

# Structured exit — downstream tasks can read this via dbutils.notebook.run()
dbutils.notebook.exit(json.dumps(result))